In [9]:
# %pip install tensorflow

In [10]:
import numpy as np
import matplotlib.pyplot as plt
from tensorflow.keras import Sequential
from tensorflow.keras.layers import Dense, LSTM, Dropout

import Analyzer
import warnings

warnings.filterwarnings("ignore")


In [11]:
mk = Analyzer.MarketDB()
raw_df = mk.get_daily_price("005930", "2023-11-14", "2026-11-12")
raw_df.head()

,code,date,open,high,low,close,diff,volume
date,,,,,,,,
2023-11-14,005930,2023-11-14,71000,71100,70600,70800,400,9567984
2023-11-15,005930,2023-11-15,71600,72200,71500,72200,1400,20148677
2023-11-16,005930,2023-11-16,72500,73000,72300,72800,600,15860451
2023-11-17,005930,2023-11-17,72300,73000,72300,72500,-300,11494644
2023-11-20,005930,2023-11-20,72100,73000,72100,72700,200,10610157


In [12]:
window_size = 10 
data_size = 5

def MinMaxScaler(data):
    """최솟값과 최댓값을 이용하여 0 ~ 1 값으로 변환"""
    numerator = data - np.min(data, 0)
    denominator = np.max(data, 0) - np.min(data, 0)
    
    # 0으로 나누기 에러가 발생하지 않도록 매우 작은 값(1e-7)을 더해서 나눔
    return numerator / (denominator + 1e-7)

dfx = raw_df[['open','high','low','volume', 'close']]
dfx = MinMaxScaler(dfx)
dfy = dfx[['close']]

x = dfx.values.tolist()
y = dfy.values.tolist()

data_x = []
data_y = []

for i in range(len(y) - window_size):
    _x = x[i : i + window_size]
    _y = y[i + window_size]

    data_x.append(_x)
    data_y.append(_y)


In [13]:
train_size = int(len(data_y) * 0.7)
train_x = np.array(data_x[0 : train_size])
train_y = np.array(data_y[0 : train_size])

test_size = len(data_y) - train_size
test_x = np.array(data_x[train_size : len(data_x)])
test_y = np.array(data_y[train_size : len(data_y)])

# 모델 생성
model = Sequential()
model.add(LSTM(units=10, activation='relu', return_sequences=True, \
               input_shape=(window_size, data_size)))
model.add(Dropout(0.1))
model.add(LSTM(units=10, activation='relu'))
model.add(Dropout(0.1))
model.add(Dense(units=1))
model.summary()

model.compile(optimizer='adam', loss='mean_squared_error')
history = model.fit(train_x, train_y, epochs=60, batch_size=30)

pred_y = model.predict(test_x)
print(len(pred_y))

print("Tomorrow's SEC price :", raw_df.close[-1], pred_y[-1], dfy.close[-1])
print("Tomorrow's SEC price :", raw_df.close[-1] * pred_y[-1] / dfy.close[-1], 'KRW')

# raw_df.close[-1] : x = dfy.close[-1] : pred_y[-1]   /  x * dfy.close[-1] = raw_df.close[-1] * pred_y[-1]
# raw_df.close[-1] * pred_y[-1] / dfy.close[-1], 'KRW')



Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ lstm (LSTM)                     │ (None, 10, 10)         │           640 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 10, 10)         │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lstm_1 (LSTM)                   │ (None, 10)             │           840 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_1 (Dropout)             │ (None, 10)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 1)              │            11 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 1,491 (5.82 KB)

 Trainable params: 1,491 (5.82 KB)

 Non-trainable params: 0 (0.00 B)

Epoch 1/60
17/17 ━━━━━━━━━━━━━━━━━━━━ 4s 16ms/step - loss: 0.0023
Epoch 2/60
17/17 ━━━━━━━━━━━━━━━━━━━━ 0s 16ms/step - loss: 0.0011
Epoch 3/60
17/17 ━━━━━━━━━━━━━━━━━━━━ 0s 21ms/step - loss: 7.2873e-04
Epoch 4/60
17/17 ━━━━━━━━━━━━━━━━━━━━ 0s 10ms/step - loss: 6.5225e-04
Epoch 5/60
17/17 ━━━━━━━━━━━━━━━━━━━━ 0s 13ms/step - loss: 3.6588e-04
Epoch 6/60
17/17 ━━━━━━━━━━━━━━━━━━━━ 0s 13ms/step - loss: 3.6676e-04
Epoch 7/60
17/17 ━━━━━━━━━━━━━━━━━━━━ 0s 11ms/step - loss: 3.3633e-04
Epoch 8/60
17/17 ━━━━━━━━━━━━━━━━━━━━ 0s 20ms/step - loss: 3.1520e-04
Epoch 9/60
17/17 ━━━━━━━━━━━━━━━━━━━━ 0s 10ms/step - loss: 2.8870e-04
Epoch 10/60
17/17 ━━━━━━━━━━━━━━━━━━━━ 0s 10ms/step - loss: 3.4427e-04
Epoch 11/60
17/17 ━━━━━━━━━━━━━━━━━━━━ 0s 9ms/step - loss: 3.3777e-04
Epoch 12/60
17/17 ━━━━━━━━━━━━━━━━━━━━ 0s 9ms/step - loss: 2.7910e-04
Epoch 13/60
17/17 ━━━━━━━━━━━━━━━━━━━━ 0s 17ms/step - loss: 3.0855e-04
Epoch 14/60
17/17 ━━━━━━━━━━━━━━━━━━━━ 0s 15ms/step - loss: 2.3012e-04
Epoch 15/60
17/17 ━━━━━━━